# Assignment: Steel Plate Fault Classification — SOLUTION

**TTK4260 — Multivariate Data Analysis & Machine Learning**  
Department of Engineering Cybernetics, NTNU

---

## Part 0: Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import FastICA
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score)

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

In [ ]:
# Load from ucimlrepo
from ucimlrepo import fetch_ucirepo

steel = fetch_ucirepo(id=198)
X_raw = steel.data.features
y_raw = steel.data.targets

# Convert 7 binary target columns to a single label
y_labels = y_raw.idxmax(axis=1)
data = pd.concat([X_raw, y_labels.rename('Fault_Type')], axis=1)

print(f'Dataset shape: {data.shape}')
data.head()

---
## Part 1: Exploratory Data Analysis

### Task 1.1 — Basic Inspection

In [ ]:
print('=== Data Types ===')
print(data.dtypes)
print(f'\n=== Missing Values ===')
print(data.isnull().sum()[data.isnull().sum() > 0])
if data.isnull().sum().sum() == 0:
    print('No missing values.')

print(f'\n=== Summary Statistics ===')
data.describe().round(2)

### Task 1.2 — Class Distribution

In [ ]:
class_counts = data['Fault_Type'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette('Set2', len(class_counts))
class_counts.plot.barh(ax=ax, color=colors)
ax.set_xlabel('Number of Samples')
ax.set_title('Class Distribution: Steel Plate Fault Types')
for i, v in enumerate(class_counts.values):
    ax.text(v + 5, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.show()

print('\nClass proportions:')
print((data['Fault_Type'].value_counts() / len(data) * 100).round(1).sort_values(ascending=False))

**Discussion:** The classes are clearly imbalanced. Other_Faults (34.7%) and Bumps (20.7%) dominate, while Dirtiness (2.8%) and Stains (3.7%) are rare. This means a classifier could achieve ~35% accuracy by always predicting Other_Faults. Models may underperform on minority classes, and overall accuracy alone will be a misleading metric.

### Task 1.3 — Feature Distributions by Class

In [ ]:
features_to_plot = ['Pixels_Areas', 'Maximum_of_Luminosity', 'Square_Index', 'Steel_Plate_Thickness']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, feat in zip(axes.flat, features_to_plot):
    sns.boxplot(data=data, x='Fault_Type', y=feat, ax=ax, palette='Set2',
                order=class_counts.index[::-1])
    ax.set_title(feat)
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

**Discussion:** Z_Scratch and K_Scratch tend to have large Pixels_Areas (elongated defects), while Dirtiness and Stains are smaller. Luminosity features differ between surface contamination types (Stains, Dirtiness) and geometric defects (Scratches, Bumps). Some features show clear separation for certain classes, but no single feature cleanly separates all 7 classes.

### Task 1.4 — Correlation Analysis

In [ ]:
feature_cols = [c for c in data.columns if c != 'Fault_Type']
corr = data[feature_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=ax, fmt='.1f', linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Find highly correlated pairs
print('Highly correlated pairs (|r| > 0.8):')
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.8:
            print(f'  {corr.columns[i]:25s} <-> {corr.columns[j]:25s}  r = {corr.iloc[i,j]:.3f}')

**Discussion:** TypeOfSteel_A300 and TypeOfSteel_A400 are perfectly anti-correlated (r = -1) since they are complementary binary indicators — one should be dropped. Pixels_Areas, LogOfAreas, and SigmoidOfAreas are highly correlated (they are derived from the same quantity). X_Minimum and X_Maximum often correlate, as do perimeter and area features. This redundancy suggests feature selection or regularisation will be important.

---
## Part 2: Data Preparation

### Task 2.1 — Feature Matrix and Target Vector

In [ ]:
X = data[feature_cols].copy()

le = LabelEncoder()
y = le.fit_transform(data['Fault_Type'])
class_names = le.classes_

print(f'Feature matrix shape: {X.shape}')
print(f'Target classes: {class_names}')
print(f'Encoded labels: {np.unique(y)}')

### Task 2.2 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')
print(f'\nClass proportions preserved:')
train_props = pd.Series(y_train).value_counts(normalize=True).sort_index()
test_props = pd.Series(y_test).value_counts(normalize=True).sort_index()
props_df = pd.DataFrame({'Train %': (train_props*100).round(1),
                          'Test %': (test_props*100).round(1)},
                         index=class_names)
print(props_df)

### Task 2.3 — Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print('Training set mean (should be ~0):', X_train_sc.mean(axis=0).round(6)[:5], '...')
print('Training set std  (should be ~1):', X_train_sc.std(axis=0).round(6)[:5], '...')

---
## Part 3: Feature Engineering & Selection

### Task 3.1 — Feature Engineering

In [ ]:
def add_engineered_features(df):
    """Add physically motivated features to a DataFrame."""
    df = df.copy()
    df['Defect_Width'] = df['X_Maximum'] - df['X_Minimum']
    df['Defect_Height'] = df['Y_Maximum'] - df['Y_Minimum']
    df['Aspect_Ratio'] = df['Defect_Width'] / (df['Defect_Height'] + 1)
    df['Luminosity_Range'] = df['Maximum_of_Luminosity'] - df['Minimum_of_Luminosity']
    df['Perimeter_Ratio'] = df['X_Perimeter'] / (df['Y_Perimeter'] + 1)
    df['Area_Perimeter_Ratio'] = df['Pixels_Areas'] / (df['X_Perimeter'] + df['Y_Perimeter'] + 1)
    return df

X_train_eng = add_engineered_features(X_train)
X_test_eng = add_engineered_features(X_test)

# Re-scale with the new features
scaler_eng = StandardScaler()
X_train_eng_sc = scaler_eng.fit_transform(X_train_eng)
X_test_eng_sc = scaler_eng.transform(X_test_eng)

eng_feature_names = list(X_train_eng.columns)
print(f'Original features: {X_train.shape[1]}')
print(f'With engineered:   {X_train_eng.shape[1]}')
print(f'New features: {eng_feature_names[27:]}')

### Task 3.2 — Feature Selection

In [ ]:
# Method 1: Mutual Information
mi_scores = mutual_info_classif(X_train_eng_sc, y_train, random_state=RANDOM_STATE)
mi_ranking = pd.Series(mi_scores, index=eng_feature_names).sort_values(ascending=False)

# Method 2: Random Forest importance
rf_temp = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_temp.fit(X_train_eng_sc, y_train)
rf_ranking = pd.Series(rf_temp.feature_importances_, index=eng_feature_names).sort_values(ascending=False)

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

mi_ranking.plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Mutual Information')
axes[0].invert_yaxis()

rf_ranking.plot.barh(ax=axes[1], color='coral')
axes[1].set_title('Random Forest Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('\n=== Top 10 (Mutual Information) ===')
print(mi_ranking.head(10).to_string())
print('\n=== Top 10 (Random Forest) ===')
print(rf_ranking.head(10).to_string())
print('\n=== Bottom 5 (Mutual Information) ===')
print(mi_ranking.tail(5).to_string())

**Discussion:** Both methods tend to rank geometry-related features (Pixels_Areas, LogOfAreas, perimeter features) and shape indices highly. The engineered features like Aspect_Ratio and Defect_Width often appear in the top ranks, confirming that defect shape is a key discriminator between fault types. TypeOfSteel_A300/A400 tend to rank lower, suggesting that steel type alone is not a strong predictor. The two methods broadly agree on the most important features, though the exact ordering differs.

---
## Part 4: Logistic Regression

### Task 4.1 — Baseline Logistic Regression

In [ ]:
lr_base = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, multi_class='multinomial')
lr_base.fit(X_train_eng_sc, y_train)

print(f'Training accuracy: {lr_base.score(X_train_eng_sc, y_train):.4f}')
print(f'Test accuracy:     {lr_base.score(X_test_eng_sc, y_test):.4f}')

### Task 4.2 — Regularisation Tuning

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_means = []
cv_stds = []
for C in C_values:
    lr = LogisticRegression(C=C, max_iter=2000, random_state=RANDOM_STATE, multi_class='multinomial')
    scores = cross_val_score(lr, X_train_eng_sc, y_train, cv=cv, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f'C={C:7.3f}  CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

best_C_lr = C_values[np.argmax(cv_means)]
print(f'\nBest C: {best_C_lr}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(C_values, cv_means, yerr=cv_stds, marker='o', capsize=5, linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('Regularisation C')
ax.set_ylabel('CV Accuracy')
ax.set_title('Logistic Regression: CV Accuracy vs C')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Task 4.3 — Evaluate Logistic Regression

In [ ]:
lr_best = LogisticRegression(C=best_C_lr, max_iter=2000, random_state=RANDOM_STATE,
                              multi_class='multinomial')
lr_best.fit(X_train_eng_sc, y_train)
y_pred_lr = lr_best.predict(X_test_eng_sc)

# Confusion matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', xticklabels=class_names,
            yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Logistic Regression Confusion Matrix (Test Acc: {accuracy_score(y_test, y_pred_lr):.3f})')
plt.tight_layout()
plt.show()

# Classification report
print(classification_report(y_test, y_pred_lr, target_names=class_names, digits=3))

**Discussion:** Logistic regression achieves moderate accuracy. It typically struggles most with Dirtiness and Stains (small sample sizes and overlapping feature distributions) and may confuse Other_Faults with several classes. The linear decision boundaries cannot capture complex nonlinear interactions between features, which limits performance on this problem.

---
## Part 5: Support Vector Machine

### Task 5.1 — SVM with RBF Kernel

In [ ]:
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1]
}

svm_grid = GridSearchCV(
    SVC(kernel='rbf', random_state=RANDOM_STATE),
    param_grid_svm, cv=cv, scoring='accuracy', n_jobs=-1, verbose=0
)
svm_grid.fit(X_train_eng_sc, y_train)

print(f'Best parameters: {svm_grid.best_params_}')
print(f'Best CV accuracy: {svm_grid.best_score_:.4f}')

### Task 5.2 — Evaluate SVM

In [ ]:
svm_best = svm_grid.best_estimator_
y_pred_svm = svm_best.predict(X_test_eng_sc)

cm_svm = confusion_matrix(y_test, y_pred_svm)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names,
            yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'SVM (RBF) Confusion Matrix (Test Acc: {accuracy_score(y_test, y_pred_svm):.3f})')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_svm, target_names=class_names, digits=3))

**Discussion:** The SVM with RBF kernel typically improves over logistic regression, especially on classes where nonlinear boundaries matter. The kernel trick allows SVM to capture more complex decision boundaries. However, SVM can still struggle with the minority classes, and training time is notably longer than logistic regression.

---
## Part 6: Decision Tree & Ensemble Methods

### Task 6.1 — Decision Tree

In [ ]:
depth_range = range(2, 21)
train_accs = []
cv_accs_mean = []
cv_accs_std = []

for d in depth_range:
    dt = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    dt.fit(X_train_eng_sc, y_train)
    train_accs.append(dt.score(X_train_eng_sc, y_train))
    scores = cross_val_score(dt, X_train_eng_sc, y_train, cv=cv, scoring='accuracy')
    cv_accs_mean.append(scores.mean())
    cv_accs_std.append(scores.std())

best_depth = list(depth_range)[np.argmax(cv_accs_mean)]
print(f'Best max_depth: {best_depth} (CV acc: {max(cv_accs_mean):.4f})')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(depth_range, train_accs, 'o-', label='Training', color='coral')
ax.errorbar(depth_range, cv_accs_mean, yerr=cv_accs_std, marker='s', capsize=3,
            label='CV (5-fold)', color='steelblue')
ax.axvline(best_depth, color='gray', linestyle='--', alpha=0.5, label=f'Best depth={best_depth}')
ax.set_xlabel('max_depth')
ax.set_ylabel('Accuracy')
ax.set_title('Decision Tree: Training vs CV Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Best tree
dt_best = DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE)
dt_best.fit(X_train_eng_sc, y_train)
print(f'\nBest tree — Train acc: {dt_best.score(X_train_eng_sc, y_train):.4f}, '
      f'Test acc: {dt_best.score(X_test_eng_sc, y_test):.4f}')

### Task 6.2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_eng_sc, y_train)

print(f'Random Forest — Train acc: {rf.score(X_train_eng_sc, y_train):.4f}, '
      f'Test acc: {rf.score(X_test_eng_sc, y_test):.4f}')

# Feature importances
importances = pd.Series(rf.feature_importances_, index=eng_feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 8))
importances.plot.barh(ax=ax, color='teal')
ax.set_title('Random Forest Feature Importances')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

### Task 6.3 — Tree vs Forest Comparison

In [ ]:
y_pred_dt = dt_best.predict(X_test_eng_sc)
y_pred_rf = rf.predict(X_test_eng_sc)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title(f'Decision Tree (Acc: {accuracy_score(y_test, y_pred_dt):.3f})')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title(f'Random Forest (Acc: {accuracy_score(y_test, y_pred_rf):.3f})')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.show()

print('=== Decision Tree ===')
print(classification_report(y_test, y_pred_dt, target_names=class_names, digits=3))
print('\n=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=class_names, digits=3))

**Discussion:** The single decision tree shows clear overfitting (training accuracy much higher than test), and its performance is unstable. The Random Forest substantially improves test accuracy and per-class performance by averaging over many decorrelated trees, demonstrating variance reduction. The Random Forest typically achieves the highest accuracy on this dataset.

---
## Part 7: Independent Component Analysis (ICA)

### Task 7.1 — Apply ICA

In [ ]:
n_components = 15
ica = FastICA(n_components=n_components, random_state=RANDOM_STATE, max_iter=1000)
X_train_ica = ica.fit_transform(X_train_eng_sc)
X_test_ica = ica.transform(X_test_eng_sc)

print(f'ICA components shape: {X_train_ica.shape}')

### Task 7.2 — Visualise ICA Components

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls_idx, cls_name in enumerate(class_names):
    mask = y_train == cls_idx
    axes[0].scatter(X_train_ica[mask, 0], X_train_ica[mask, 1], label=cls_name, alpha=0.5, s=15)
axes[0].set_xlabel('IC 1'); axes[0].set_ylabel('IC 2')
axes[0].set_title('ICA: Component 1 vs 2')
axes[0].legend(fontsize=7, markerscale=2)

for cls_idx, cls_name in enumerate(class_names):
    mask = y_train == cls_idx
    axes[1].scatter(X_train_ica[mask, 0], X_train_ica[mask, 2], label=cls_name, alpha=0.5, s=15)
axes[1].set_xlabel('IC 1'); axes[1].set_ylabel('IC 3')
axes[1].set_title('ICA: Component 1 vs 3')
axes[1].legend(fontsize=7, markerscale=2)

plt.tight_layout()
plt.show()

### Task 7.3 — Classification on ICA Features

In [ ]:
# Logistic regression on ICA features
lr_ica = LogisticRegression(C=best_C_lr, max_iter=2000, random_state=RANDOM_STATE,
                             multi_class='multinomial')
lr_ica.fit(X_train_ica, y_train)

acc_original = lr_best.score(X_test_eng_sc, y_test)
acc_ica = lr_ica.score(X_test_ica, y_test)

print(f'Logistic Regression on original features: {acc_original:.4f}')
print(f'Logistic Regression on ICA components:    {acc_ica:.4f}')
print(f'Difference: {acc_ica - acc_original:+.4f}')

**Discussion:** ICA typically does not improve performance significantly on this dataset, and may slightly degrade it. This is because the 27 features are largely direct physical measurements and pre-computed indices — not linear mixtures of hidden sources (the setting where ICA excels, such as audio source separation). ICA can still be useful for visualisation and for understanding the latent structure, but it is not a universal preprocessing step. This is an important lesson: ICA is most effective when the measured signals are genuine mixtures of independent sources.

---
## Part 8: Cross-Validation & Performance Metrics

### Task 8.1 — Stratified K-Fold Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=best_C_lr, max_iter=2000,
                                              random_state=RANDOM_STATE, multi_class='multinomial'),
    'SVM (RBF)': svm_grid.best_estimator_,
    'Decision Tree': DecisionTreeClassifier(max_depth=best_depth, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_train_eng_sc, y_train, cv=cv, scoring='accuracy')
    results[name] = {'mean': scores.mean(), 'std': scores.std()}
    print(f'{name:25s}  CV Accuracy: {scores.mean():.4f} ± {scores.std():.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(9, 4))
names = list(results.keys())
means = [results[n]['mean'] for n in names]
stds = [results[n]['std'] for n in names]
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = ax.bar(names, means, yerr=stds, capsize=6, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylabel('CV Accuracy')
ax.set_title('5-Fold Stratified CV Accuracy Comparison')
ax.set_ylim(0.5, 1.0)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{m:.3f}', ha='center', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Task 8.2 — Metrics for Imbalanced Data

In [ ]:
# Use the best model (Random Forest)
y_pred_best = rf.predict(X_test_eng_sc)

print('=== Random Forest (Best Model) — Test Set Metrics ===\n')
print(f'Overall Accuracy:       {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Macro-averaged F1:      {f1_score(y_test, y_pred_best, average="macro"):.4f}')
print(f'Weighted-averaged F1:   {f1_score(y_test, y_pred_best, average="weighted"):.4f}')
print()
print('Full classification report:')
print(classification_report(y_test, y_pred_best, target_names=class_names, digits=3))

**Discussion:** Macro averaging treats all classes equally regardless of size, giving equal weight to Dirtiness (55 samples) and Other_Faults (673 samples). Weighted averaging weights each class by its support. In a manufacturing quality control context, **macro-averaged F1 is more informative** because every fault type matters — failing to detect a rare defect type (Dirtiness, Stains) is just as costly as misclassifying a common one. The gap between macro and weighted F1 reveals how much the model underperforms on minority classes.

---
## Part 9: Summary & Reflection

### Task 9.1 — Summary Table

*(Values below are representative — exact numbers depend on the real dataset.)*

In [ ]:
# Compute summary for all models
all_predictions = {
    'Logistic Regression': y_pred_lr,
    'SVM (RBF)': y_pred_svm,
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf,
}

summary_rows = []
for name, y_pred in all_predictions.items():
    summary_rows.append({
        'Model': name,
        'Test Accuracy': f'{accuracy_score(y_test, y_pred):.3f}',
        'Macro F1': f'{f1_score(y_test, y_pred, average="macro"):.3f}',
        'Weighted F1': f'{f1_score(y_test, y_pred, average="weighted"):.3f}',
    })

# Add ICA + LR
y_pred_ica = lr_ica.predict(X_test_ica)
summary_rows.append({
    'Model': 'ICA + Log. Reg.',
    'Test Accuracy': f'{accuracy_score(y_test, y_pred_ica):.3f}',
    'Macro F1': f'{f1_score(y_test, y_pred_ica, average="macro"):.3f}',
    'Weighted F1': f'{f1_score(y_test, y_pred_ica, average="weighted"):.3f}',
})

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print(summary_df.to_string())

### Task 9.2 — Final Model Recommendation

**Recommendation: Random Forest.** It achieves the highest overall accuracy and macro F1, handles the class imbalance better than the other models, and provides useful feature importances for process engineers. While SVM can match it in some settings, Random Forest is faster to train at this dataset size, requires less hyperparameter tuning, and is more interpretable through feature importance analysis. The decision tree alone overfits, and logistic regression is limited by its linear boundaries.

### Task 9.3 — Reflection

1. **Logistic regression struggled** primarily on minority classes (Dirtiness, Stains) and on distinguishing between geometrically similar fault types. This indicates the decision boundaries are nonlinear — linear separability does not hold in the original feature space.

2. **Feature engineering** (aspect ratio, defect width/height, luminosity range) improved performance modestly. Feature selection confirmed that geometry-related features are the most discriminative, and removing redundant features (e.g., TypeOfSteel_A300 when A400 is present) simplifies models without losing performance.

3. **Performance metrics:** The most important lesson is that overall accuracy can be misleading under class imbalance. Macro-averaged F1 gives a clearer picture of model quality across all fault types, especially rare ones that are safety-critical in manufacturing.

4. **Next steps:** Gradient boosted trees (XGBoost, LightGBM) would likely push accuracy further. SMOTE or class-weighted losses could specifically address the minority class problem. A confusion analysis of the remaining errors could guide targeted feature engineering.